In [8]:
sample_detections = [
    {"frame": 118, "class": "D40_pothole", "conf": 0.71, "gps": (13.0450, 80.2110), "bbox": [120,200,180,260]},
    {"frame": 119, "class": "D40_pothole", "conf": 0.88, "gps": (13.0451, 80.2111), "bbox": [122,198,182,258]},
    {"frame": 120, "class": "D40_pothole", "conf": 0.94, "gps": (13.0452, 80.2112), "bbox": [124,199,184,259]},
    {"frame": 121, "class": "D40_pothole", "conf": 0.92, "gps": (13.0453, 80.2113), "bbox": [125,200,185,260]},
    {"frame": 200, "class": "D00_longitudinal_crack", "conf": 0.55, "gps": (13.0600, 80.2200), "bbox": [50,50,90,55]},
]

In [9]:
from math import radians, sin, cos, sqrt, atan2

def gps_distance_meters(gps1, gps2):
    R = 6371000
    lat1, lon1 = radians(gps1[0]), radians(gps1[1])
    lat2, lon2 = radians(gps2[0]), radians(gps2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def verify_incidents(detections, min_conf=0.6, max_frame_gap=5, max_gps_dist=15, min_streak=3):
    detections = sorted(detections, key=lambda d: d["frame"])
    incidents = []
    current_group = []

    for det in detections:
        if det["conf"] < min_conf:
            continue
        if not current_group:
            current_group = [det]
            continue

        last = current_group[-1]
        same_class = det["class"] == last["class"]
        close_frame = (det["frame"] - last["frame"]) <= max_frame_gap
        close_gps = gps_distance_meters(det["gps"], last["gps"]) <= max_gps_dist

        if same_class and close_frame and close_gps:
            current_group.append(det)
        else:
            if len(current_group) >= min_streak:
                incidents.append(build_incident(current_group))
            current_group = [det]

    if len(current_group) >= min_streak:
        incidents.append(build_incident(current_group))

    return incidents

def build_incident(group):
    avg_conf = sum(d["conf"] for d in group) / len(group)
    return {
        "incident_id": f"RA-{group[0]['frame']:04d}",
        "anomaly": group[0]["class"],
        "status": "VERIFIED",
        "evidence_frames": len(group),
        "avg_confidence": round(avg_conf, 3),
        "gps": group[len(group)//2]["gps"],
        "frame_range": (group[0]["frame"], group[-1]["frame"])
    }

incidents = verify_incidents(sample_detections)
for inc in incidents:
    print(inc)

In [11]:
print("Total sample detections:", len(sample_detections))
incidents = verify_incidents(sample_detections)
print("Number of incidents found:", len(incidents))
print(incidents)

Total sample detections: 5
Number of incidents found: 0
[]


In [12]:
det1 = sample_detections[0]
det2 = sample_detections[1]

print("Same class:", det1["class"] == det2["class"])
print("Frame gap:", det2["frame"] - det1["frame"])
print("GPS distance (m):", gps_distance_meters(det1["gps"], det2["gps"]))

Same class: True
Frame gap: 1
GPS distance (m): 15.523750792539847


In [13]:
incidents = verify_incidents(sample_detections, max_gps_dist=20)
for inc in incidents:
    print(inc)

{'incident_id': 'RA-0118', 'anomaly': 'D40_pothole', 'status': 'VERIFIED', 'evidence_frames': 4, 'avg_confidence': 0.863, 'gps': (13.0452, 80.2112), 'frame_range': (118, 121)}
